<h1>Pandas Part 4</h1>


In [77]:
import pandas as pd
import numpy as np

## Cleaned City Data

In module 4.3 we cleaned up the European city data by hand. I saved the cleaned data in a new csv file.

### to_csv()

I used the DataFrame method to_csv() to create the improved csv file.  Recall that in 4.3 we called the DataFrame df.  After making the changes use the command `df.to_csv('../data/ec.csv',index=False)` to write the new file ec.csv. **index=False** means that the index values will not be written to the csv file.


Here are the first few lines of the file I created with to_csv():

In [78]:
!head -n 5 ../data/ec.csv

Rank,City,Country,Population,Date
1,London[2],United Kingdom,8615246,2014-06-01
2,Berlin,Germany,3437916,2014-05-31
3,Madrid,Spain,3165235,2014-01-01
4,Rome,Italy,2872086,2014-09-30


This time when I read the csv file I will attempt to have the date strings parsed to datetime object by including the optional argument **parse_dates=\[4]**.

In stead of using the index 4, I could use the name of the column **parse_dates=['Date']**.

I will use **index_col=0** to make the column named Rank the index.

In [79]:
cities = pd.read_csv('../data/ec.csv',parse_dates=[4],index_col=0)

cities.head()

,City,Country,Population,Date
Rank,,,,
1,London[2],United Kingdom,8615246,2014-06-01
2,Berlin,Germany,3437916,2014-05-31
3,Madrid,Spain,3165235,2014-01-01
4,Rome,Italy,2872086,2014-09-30
5,Paris,France,2273305,2013-01-01


In [80]:
cities.info()

<class 'pandas.core.frame.DataFrame'>
Index: 105 entries, 1 to 105
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   City        105 non-null    object        
 1   Country     105 non-null    object        
 2   Population  105 non-null    int64         
 3   Date        105 non-null    datetime64[ns]
dtypes: datetime64[ns](1), int64(1), object(2)
memory usage: 4.1+ KB


The cities DataFrame is clean and ready.

### read_csv(): optional arguments

I mentioned that there are optional arguments that fix most of the things we fixed by hand.

1. thousands = ','  This argument removes the commas from the population values and parses them as integers.


2. skipinitialspace = True  This argument removes the spaces the strings.

Next I read the original csv file with the optional arguments.

In [81]:
cities_orig = pd.read_csv('../data/european_cities.csv',index_col=0,thousands=',',
                          parse_dates=[4],skipinitialspace=True)
cities_orig.head()

,City,State,Population,Date of census/estimate
Rank,,,,
1,London[2],United Kingdom,8615246,2014-06-01
2,Berlin,Germany,3437916,2014-05-31
3,Madrid,Spain,3165235,2014-01-01
4,Rome,Italy,2872086,2014-09-30
5,Paris,France,2273305,2013-01-01


In [82]:
cities_orig.info()

<class 'pandas.core.frame.DataFrame'>
Index: 105 entries, 1 to 105
Data columns (total 4 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   City                     105 non-null    object        
 1   State                    105 non-null    object        
 2   Population               105 non-null    int64         
 3   Date of census/estimate  105 non-null    datetime64[ns]
dtypes: datetime64[ns](1), int64(1), object(2)
memory usage: 4.1+ KB


Now use .values to get the numpy array and check for spaces before the country names:

In [83]:
cities_orig.values[:3]

array([['London[2]', 'United Kingdom', 8615246,
        Timestamp('2014-06-01 00:00:00')],
       ['Berlin', 'Germany', 3437916, Timestamp('2014-05-31 00:00:00')],
       ['Madrid', 'Spain', 3165235, Timestamp('2014-01-01 00:00:00')]],
      dtype=object)

The spaces before the country names have been removed.

### rename()

I could have used the optional argument `columns=['City','Country','Population','Date']` to fix the column names, but I want to show you how to use the rename() function.

We have seen how to use a list to rename all the columns, but if you have a lot of columns and you want to rename just a few, that method is cumbersome.

The rename() method allows you to rename some or all of the columns.

In rename() I make the argument columns a dictionary where the keys are the column names I want to change and the values are the new column names.  I also use `inplace=True` to make the changes instead of just showing a view.

In [84]:
cities_orig.columns[3]

'Date of census/estimate'

In [85]:
cities_orig.rename(columns={cities_orig.columns[3]:'Date','State':'Country'},inplace=True)
cities_orig.head()

,City,Country,Population,Date
Rank,,,,
1,London[2],United Kingdom,8615246,2014-06-01
2,Berlin,Germany,3437916,2014-05-31
3,Madrid,Spain,3165235,2014-01-01
4,Rome,Italy,2872086,2014-09-30
5,Paris,France,2273305,2013-01-01


**Note** the first key in the columns dictionary, cities_orig.columns\[3].  I didn't want to type the original long column name, so I pulled it from the cities_orig.columns list.

There is a DataFrame method named **equals()** that checks to see if two DataFrames have the same data.

In [86]:
cities.equals(cities_orig)

True

## value_counts()

value_counts() is a Series method that returns the unique items in the Series along with the number of times each item occurs.  By default it sorts the results from greatest number to smallest.


Calling value_counts() on the Series cities.Country will tell us how many times each country occurs, and that is the same as the number of cities from that country that are in the data.

In [87]:
cities.Country.value_counts()

Country
Germany                     19
United Kingdom              16
Spain                       13
Poland                      10
Italy                       10
France                       5
Netherlands                  4
Bulgaria                     3
Sweden                       3
Romania                      3
Czech Republic               2
Belgium                      2
Greece                       2
Denmark                      2
Lithuania                    2
Hungary                      1
Austria                      1
Croatia                      1
Latvia                       1
Finland                      1
Portugal                     1
Ireland                      1
Estonia                      1
Slovakia Slovak Republic     1
Name: count, dtype: int64

The value_counts() method is **used a lot!**

## groupby()

The groupby() method is similar to GROUP BY in SQL.  It splits the DataFrame into chunks called groups.

First I build a simple DataFrame to illustrate groupby().

**Note:** In this notebook I will save the output of the groupby() command and examine its features.  In general, you do not save a groupby object, you simply follow groupby() with an aggregate function.

In [88]:
d = np.array([['CAS','CAS','BUS','BUS','CAS','CAS','BUS','CAS','BUS'],
              ['MATH','HISTORY','ACCOUNTING','FINANCE','MATH','MATH','FINANCE','HISTORY','ACCOUNTING']])

s = pd.DataFrame(d.T,columns=['School','Dept'])
s['Class'] = ['stt215','hst101','acg101','fin235','mat475','mat592','fin101','hst201','acg201']
s['Enrollment'] = [45,25,30,35,10,12,25,20,25]
s['ContactHours'] = [5,3,4,3,4,4,3,3,4]
s

,School,Dept,Class,Enrollment,ContactHours
0,CAS,MATH,stt215,45,5
1,CAS,HISTORY,hst101,25,3
2,BUS,ACCOUNTING,acg101,30,4
3,BUS,FINANCE,fin235,35,3
4,CAS,MATH,mat475,10,4
5,CAS,MATH,mat592,12,4
6,BUS,FINANCE,fin101,25,3
7,CAS,HISTORY,hst201,20,3
8,BUS,ACCOUNTING,acg201,25,4


### Examining Groups

I am going to group by School.  There are two schools, CAS and BUS, so there will be two groups named CAS and BUS.

In practice, you generally see the groupby() method followed by a call to a function that will act on the groups. You rarely see a groupby object used the way I will use it here, but I think this will help you understand groupby().

In [89]:
g = s.groupby('School')

g is not a DataFrame, it is a DataFrameGroupBy object.  The method get_group('BUS') returns the BUS group as a DataFrame.

In [90]:
g.get_group('BUS')

,School,Dept,Class,Enrollment,ContactHours
2,BUS,ACCOUNTING,acg101,30,4
3,BUS,FINANCE,fin235,35,3
6,BUS,FINANCE,fin101,25,3
8,BUS,ACCOUNTING,acg201,25,4


Use a for loop to print all the groups.

When you use a groupby object in a loop, the iterations return a **tuple**: name of group followed by data.

In [91]:
for groupName,groupData in g:
    print(groupName,'\n')
    print(groupData,'\n\n')

BUS 

  School        Dept   Class  Enrollment  ContactHours
2    BUS  ACCOUNTING  acg101          30             4
3    BUS     FINANCE  fin235          35             3
6    BUS     FINANCE  fin101          25             3
8    BUS  ACCOUNTING  acg201          25             4 


CAS 

  School     Dept   Class  Enrollment  ContactHours
0    CAS     MATH  stt215          45             5
1    CAS  HISTORY  hst101          25             3
4    CAS     MATH  mat475          10             4
5    CAS     MATH  mat592          12             4
7    CAS  HISTORY  hst201          20             3 




As in SQL, you generally want to call an aggregate function after grouping.

If the groups have just one column, the result of calling an aggregate function is usually a Series. With multiple columns the result will be a DataFrame.

Whether the result is a Series or a DataFrame, the default behavior is that **what you group by becomes the index.**

If you set the optional argument `as_index=False`, then the column grouped by does not become the index.

### Aggregate Functions

In previous versions of pandas, when you call a numeric aggregate function, such as sum(), min(), max(), mean(), or median(), on a groupby object, it acts on the columns that are numeric.

Now pandas tried to apply these numeric functions to all columns. So, first narrow the scope to include only numeric columns.

In [92]:
# sum of the enrollments and contact hours by School

g[['Enrollment','ContactHours']].sum()

,Enrollment,ContactHours
School,,
BUS,115,14
CAS,112,19


This is the result if you use `as_index=False`:

In [93]:
s.groupby('School',as_index=False)[['Enrollment','ContactHours']].sum()

,School,Enrollment,ContactHours
0,BUS,115,14
1,CAS,112,19


The groupby method **size()** returns the number of rows in each group.

In [94]:
g.size()

School
BUS    4
CAS    5
dtype: int64

The method **count()** can act on any datatype, so calling g.count() returns a count for every column.

In [95]:
g.count()

,Dept,Class,Enrollment,ContactHours
School,,,,
BUS,4,4,4,4
CAS,5,5,5,5


In this case we get the same count for each column, but if the DataFrame s had null values, then the counts could be different for different columns.

### Exercises

The following example came from this tutorial: https://www.machinelearningplus.com/pandas/pandas-groupby-examples/

In [96]:
# Create the data of the DataFrame as a dictionary
data_df = {'Name': ['Asha', 'Harsh', 'Sourav', 'Riya', 'Hritik',
                    'Shivansh', 'Rohan', 'Akash', 'Soumya', 'Kartik'],

           'Department': ['Administration', 'Marketing', 'Technical', 'Technical', 'Marketing',
                          'Administration', 'Technical', 'Marketing', 'Technical', 'Administration'],

           'Employment Type': ['Full-time Employee', 'Intern', 'Intern', 'Part-time Employee', 'Part-time Employee',
                               'Full-time Employee', 'Full-time Employee', 'Intern', 'Intern', 'Full-time Employee'],

           'Salary': [120000, 50000, 70000, 70000, 55000,
                      120000, 125000, 60000, 50000, 120000],

           'Years of Experience': [5, 1, 2, 3, 4,
                                   7, 6, 2, 1, 6]}

# Create the DataFrame
mlp = pd.DataFrame(data_df)
mlp

,Name,Department,Employment Type,Salary,Years of Experience
0,Asha,Administration,Full-time Employee,120000,5
1,Harsh,Marketing,Intern,50000,1
2,Sourav,Technical,Intern,70000,2
3,Riya,Technical,Part-time Employee,70000,3
4,Hritik,Marketing,Part-time Employee,55000,4
5,Shivansh,Administration,Full-time Employee,120000,7
6,Rohan,Technical,Full-time Employee,125000,6
7,Akash,Marketing,Intern,60000,2
8,Soumya,Technical,Intern,50000,1
9,Kartik,Administration,Full-time Employee,120000,6


#### Exercise

Group by Department, narrow focus to 'Years of Experience', and find the mean().

Partial Answer: in Marketing, the mean years of experience is 2.333333.

In [97]:
mlp.groupby('Department')['Years of Experience'].mean()


Department
Administration    6.000000
Marketing         2.333333
Technical         3.000000
Name: Years of Experience, dtype: float64

#### Exercise

Find the maximum salary for each department.

Partial Answer: the maximum salary in the Technical department is 125,000.

In [98]:
mlp.groupby('Department')['Salary'].max()


Department
Administration    120000
Marketing          60000
Technical         125000
Name: Salary, dtype: int64

### aggregate() or agg() for short

The **aggregate()** method, **agg()** for short, lets you call multiple aggregate functions.

**Example**

In [99]:
df1 = g[['Enrollment','ContactHours']].agg(['min','max','mean','median','count'])
df1

Enrollment                         ContactHours                      
              min max   mean median count          min max mean median count
School                                                                      
BUS            25  35  28.75   27.5     4            3   4  3.5    3.5     4
CAS            10  45  22.40   20.0     5            3   5  3.8    4.0     5

### MultiIndex

Look at the column names in df1.  Because we used multiple aggregate functions, the columns have a hierarchical index called a MultiIndex.

In [100]:
df1.columns

MultiIndex([(  'Enrollment',    'min'),
            (  'Enrollment',    'max'),
            (  'Enrollment',   'mean'),
            (  'Enrollment', 'median'),
            (  'Enrollment',  'count'),
            ('ContactHours',    'min'),
            ('ContactHours',    'max'),
            ('ContactHours',   'mean'),
            ('ContactHours', 'median'),
            ('ContactHours',  'count')],
           )

If you want to see just the 'max' column of df1, **df1\['max'] does not work**.

The full name of the column is the tuple ('Enrollment','max'), so use **df1[('Enrollment','max')]**

In [101]:
df1[('Enrollment','max')]

School
BUS    35
CAS    45
Name: (Enrollment, max), dtype: int64

### Group By Multiple Fields

You can group by more than one field.  Here I group first by School, then within School by Dept. This will create four groups.  You can see the groups by using a loop.

In [102]:
gg = s.groupby(['School','Dept'])

for name,data in gg:
    print(name,'\n')
    print(data,'\n\n')

('BUS', 'ACCOUNTING') 

  School        Dept   Class  Enrollment  ContactHours
2    BUS  ACCOUNTING  acg101          30             4
8    BUS  ACCOUNTING  acg201          25             4 


('BUS', 'FINANCE') 

  School     Dept   Class  Enrollment  ContactHours
3    BUS  FINANCE  fin235          35             3
6    BUS  FINANCE  fin101          25             3 


('CAS', 'HISTORY') 

  School     Dept   Class  Enrollment  ContactHours
1    CAS  HISTORY  hst101          25             3
7    CAS  HISTORY  hst201          20             3 


('CAS', 'MATH') 

  School  Dept   Class  Enrollment  ContactHours
0    CAS  MATH  stt215          45             5
4    CAS  MATH  mat475          10             4
5    CAS  MATH  mat592          12             4 




gg.size() gives you the number of classes in each of the four groups.  It returns a Series with a multiindex.

In [103]:
gs = gg.size()
gs

School  Dept      
BUS     ACCOUNTING    2
        FINANCE       2
CAS     HISTORY       2
        MATH          3
dtype: int64

You can select one value from gs using the complete index value as a tuple.

In [104]:
gs[('CAS','MATH')]

3

You can get both items in 'CAS' just using 'CAS'

In [105]:
gs['CAS']

Dept
HISTORY    2
MATH       3
dtype: int64

You cannot get the ('CAS','MATH') item with just 'MATH'.

In [106]:
gs['MATH']

KeyError: 'MATH'

### Exercises

These exercises use the mlp DataFrame defined in the first exercises group.


In [107]:
mlp

,Name,Department,Employment Type,Salary,Years of Experience
0,Asha,Administration,Full-time Employee,120000,5
1,Harsh,Marketing,Intern,50000,1
2,Sourav,Technical,Intern,70000,2
3,Riya,Technical,Part-time Employee,70000,3
4,Hritik,Marketing,Part-time Employee,55000,4
5,Shivansh,Administration,Full-time Employee,120000,7
6,Rohan,Technical,Full-time Employee,125000,6
7,Akash,Marketing,Intern,60000,2
8,Soumya,Technical,Intern,50000,1
9,Kartik,Administration,Full-time Employee,120000,6


#### Exercise

Find the min, mean, and max of the Salary and Years of Experience by department.

Partial Answer: In Marketing, the min salary is 50,000 and the max years of experience is 4.

In [108]:
mlp.groupby('Department')[['Salary', 'Years of Experience']].agg(['min', 'mean', 'max'])

Salary                   Years of Experience              
                   min      mean     max                 min      mean max
Department                                                                
Administration  120000  120000.0  120000                   5  6.000000   7
Marketing        50000   55000.0   60000                   1  2.333333   4
Technical        50000   78750.0  125000                   1  3.000000   6

#### Exercise

Find the mean salary by Department, then by Employment Type.

Partial Answer: in the Technical department, the mean salary of Interns is 60,000.

In [109]:
mlp.groupby(['Department', 'Employment Type'])['Salary'].mean()

Department      Employment Type   
Administration  Full-time Employee    120000.0
Marketing       Intern                 55000.0
                Part-time Employee     55000.0
Technical       Full-time Employee    125000.0
                Intern                 60000.0
                Part-time Employee     70000.0
Name: Salary, dtype: float64

### Dict argument to agg()

You can give a dictionary object to the agg() method to specify how to treat different columns.<p>
The dictionary {'Class':'count','Enrollment':'mean'} says to count the classes in each of the four groups, and to average the class sizes.

In [110]:
gg.agg({'Class':'count','Enrollment':'mean'})

Class  Enrollment
School Dept                         
BUS    ACCOUNTING      2   27.500000
       FINANCE         2   30.000000
CAS    HISTORY         2   22.500000
       MATH            3   22.333333

### Specifying Aggregate Functions

We have been using strings to determine the aggregate function, for example 'mean' and 'count'.

Sometimes you see a function object used instead of a string.  You could use np.mean in place of 'mean' and the result will be the same.

In [111]:
gg.agg({'Class':'count','Enrollment':np.mean})

Class  Enrollment
School Dept                         
BUS    ACCOUNTING      2   27.500000
       FINANCE         2   30.000000
CAS    HISTORY         2   22.500000
       MATH            3   22.333333

### Using groupby() to replace value_counts()

At the beginning of this module I used value_counts() to find the number of cities in each country in the European cities data set.

**Value_counts() is the simplest way that I know to find that information, but to provide another example I am going to get the same information using groupby().**

This will probably make you appreciate value_counts().

Here is the plan:

<ul>
    <li>First group by Country.</li>
    <li>Restrict attention to the City column.</li>
    <li>Call the count() method.  You could use size() instead of count() and get the same result.</li>
    <li>Sort the values from high to low.  <b>Note</b>:  When you restrict to one column, you have a Series, and sort_values() on a Series, unlike a DataFrame, does not need to be told what column to sort on.</li>
</ul>

In [112]:
ec.groupby('Country')['City'].size().sort_values(ascending=False)

NameError: name 'ec' is not defined

The following DataFrame will be used in exercises.

In [113]:
emp = pd.read_csv('../data/emp.csv')
emp

,name,dept,sex,years,salary
0,Sue,admin,f,23,148000
1,Joe,maintenance,m,5,57000
2,Sam,operations,m,15,90000
3,Beth,maintenance,f,7,61000
4,George,admin,m,15,122000
5,James,operations,m,7,82000
6,Sally,admin,f,12,93000
7,Mary,operations,f,6,80000
8,Jim,operations,m,1,73000
9,Julie,maintenance,f,8,63000


In [114]:
emp

,name,dept,sex,years,salary
0,Sue,admin,f,23,148000
1,Joe,maintenance,m,5,57000
2,Sam,operations,m,15,90000
3,Beth,maintenance,f,7,61000
4,George,admin,m,15,122000
5,James,operations,m,7,82000
6,Sally,admin,f,12,93000
7,Mary,operations,f,6,80000
8,Jim,operations,m,1,73000
9,Julie,maintenance,f,8,63000


## Exercises
    
    
These exercises refer to the emp dataframe defined above.
<ol>
<li>Use groupby() to find the number of females and males.  Answer: 5 and 7</li><p/>
<li>Use groupby() to find the number of females and males by department.<p/>Partial Answer: maintenance and operations departments each have 3 males.</li><p/>
<li>Find the minimum, maximum and average salary by department.<p/>Partial Answer: Min salary in admin is 93,000 and mean salary in operations is 81,250.</li><p/>
<li>Find the average years of service and average salary first by department then by sex.<p/>Partial Answer: in maintenance the mean salary for females is 62,000 and for males it is 57,000.</li>
<ol>

In [115]:
emp.groupby('sex').size()

sex
f    5
m    7
dtype: int64

In [116]:
emp.groupby(['dept', 'sex']).size()

dept         sex
admin        f      2
             m      1
maintenance  f      2
             m      3
operations   f      1
             m      3
dtype: int64

In [117]:
emp.groupby('dept')['salary'].agg(['min', 'max', 'mean'])

,min,max,mean
dept,,,
admin,93000,148000,121000.0
maintenance,54000,63000,59000.0
operations,73000,90000,81250.0


In [118]:
emp.groupby(['dept', 'sex'])[['years', 'salary']].mean()

years         salary
dept        sex                          
admin       f    17.500000  120500.000000
            m    15.000000  122000.000000
maintenance f     7.500000   62000.000000
            m     5.000000   57000.000000
operations  f     6.000000   80000.000000
            m     7.666667   81666.666667